In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from torch import nn

SEED = 42
BATCH_SIZE = 2048
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "data" / "creditcard.csv").exists()
        and (path / "results" / "split_indices.npz").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate project data and results/split_indices.npz. "
        "Run Notebook 01 first."
    )

RESULTS_DIR = PROJECT_ROOT / "results"
required_files = [
    "statistical_baselines.json",
    "isolation_forest.joblib",
    "isolation_forest_config.json",
    "autoencoder_state.pt",
    "autoencoder_scaler.joblib",
    "autoencoder_config.json",
    "random_forest_config.json",
]
missing_files = [name for name in required_files if not (RESULTS_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        "Run the corresponding baseline/model notebooks first; missing files: "
        + ", ".join(missing_files)
    )

print("Loading dataset...", flush=True)
df = pd.read_csv(PROJECT_ROOT / "data" / "creditcard.csv")
with np.load(RESULTS_DIR / "split_indices.npz") as split_indices:
    train_idx = split_indices["train_idx"]
    validation_idx = split_indices["validation_idx"]
train_df = df.loc[train_idx].copy()
validation_df = df.loc[validation_idx].copy()
y_val = validation_df["Class"].to_numpy()

with (RESULTS_DIR / "statistical_baselines.json").open(encoding="utf-8") as file:
    statistical_record = json.load(file)
with (RESULTS_DIR / "isolation_forest_config.json").open(encoding="utf-8") as file:
    isolation_config = json.load(file)
with (RESULTS_DIR / "autoencoder_config.json").open(encoding="utf-8") as file:
    autoencoder_config = json.load(file)
with (RESULTS_DIR / "random_forest_config.json").open(encoding="utf-8") as file:
    random_forest_record = json.load(file)

feature_cols = statistical_record["feature_cols"]
if isolation_config.get("feature_cols") != feature_cols:
    raise ValueError("Isolation Forest feature columns do not match the baseline features.")
if autoencoder_config.get("feature_cols") != feature_cols:
    raise ValueError("Autoencoder feature columns do not match the baseline features.")
if random_forest_record.get("feature_cols") != feature_cols:
    raise ValueError("Random Forest feature columns do not match the baseline features.")
X_val = validation_df[feature_cols]

print("Calculating stat scores...", flush=True)
reference = statistical_record["reference_statistics"]
train_mean = pd.Series(reference["mean"]).reindex(feature_cols)
train_std = pd.Series(reference["std"]).reindex(feature_cols)
q1 = pd.Series(reference["q1"]).reindex(feature_cols)
q3 = pd.Series(reference["q3"]).reindex(feature_cols)
if any(values.isna().any() for values in (train_mean, train_std, q1, q3)):
    raise ValueError("Saved statistical reference values are incomplete.")
z_deviations = (X_val - train_mean).abs().div(train_std.replace(0, np.nan))
z_flags = z_deviations.gt(3) | (train_std.eq(0) & X_val.ne(train_mean))
z_scores = z_flags.fillna(False).mean(axis=1).to_numpy()
iqr = q3 - q1
iqr_flags = X_val.lt(q1 - 1.5 * iqr) | X_val.gt(q3 + 1.5 * iqr)
iqr_scores = iqr_flags.mean(axis=1).to_numpy()

print("Scoring Isolation Forest...", flush=True)
isolation_forest = joblib.load(RESULTS_DIR / "isolation_forest.joblib")
isolation_scores = -isolation_forest.decision_function(X_val)


autoencoder_scaler = joblib.load(RESULTS_DIR / "autoencoder_scaler.joblib")
X_val_scaled = autoencoder_scaler.transform(X_val).astype(np.float32)

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim),
        )

    def forward(self, values):
        return self.network(values)


def reconstruction_scores(model, values):
    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(values), BATCH_SIZE):
            batch = torch.from_numpy(values[start : start + BATCH_SIZE])
            scores.append((model(batch) - batch).pow(2).mean(dim=1).numpy())
    return np.concatenate(scores)

print("Scoring autoencoder...", flush=True)
autoencoder = Autoencoder(len(feature_cols))
autoencoder.load_state_dict(
    torch.load(RESULTS_DIR / "autoencoder_state.pt", map_location="cpu")
)
autoencoder_scores = reconstruction_scores(autoencoder, X_val_scaled)

print("Combining scores and selecting threshold...", flush=True)
score_components = {
    "z_score": z_scores,
    "iqr": iqr_scores,
    "isolation_forest": isolation_scores,
    "autoencoder": autoencoder_scores,
}
np.savez_compressed(
    RESULTS_DIR / "hybrid_rank_reference.npz",
    **{name: np.sort(scores) for name, scores in score_components.items()},
)
normalized_scores = pd.DataFrame(
    {
        name: pd.Series(scores).rank(method="average", pct=True).to_numpy()
        for name, scores in score_components.items()
    }
)
hybrid_scores = normalized_scores.mean(axis=1).to_numpy()


def evaluate_scores(y_true, scores, threshold):
    predictions = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    }


precision, recall, thresholds = precision_recall_curve(y_val, hybrid_scores)
if len(thresholds) == 0:
    raise ValueError("No hybrid validation thresholds available.")
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)
best_index = int(np.argmax(f1_scores))
hybrid_threshold = float(thresholds[best_index])

validation_results = {
    "z_score": evaluate_scores(
        y_val, z_scores, statistical_record["validation_results"]["z_score"]["threshold"]
    ),
    "iqr": evaluate_scores(
        y_val, iqr_scores, statistical_record["validation_results"]["iqr"]["threshold"]
    ),
    "isolation_forest": evaluate_scores(
        y_val, isolation_scores, isolation_config["threshold"]
    ),
    "autoencoder": evaluate_scores(
        y_val, autoencoder_scores,
        autoencoder_config["validation_metrics"]["threshold"],
    ),
    "hybrid": evaluate_scores(y_val, hybrid_scores, hybrid_threshold),
    "random_forest": random_forest_record["validation_metrics"],
}
comparison = {
    "random_seed": SEED,
    "score_normalization": "Validation-split percentile ranks",
    "hybrid_components": list(score_components),
    "hybrid_weights": {name: 1 / len(score_components) for name in score_components},
    "validation_results": validation_results,
}
with (RESULTS_DIR / "validation_comparison.json").open("w", encoding="utf-8") as file:
    json.dump(comparison, file, indent=2)
print(json.dumps(comparison, indent=2))
print(f"Saved validation comparison to {RESULTS_DIR / 'validation_comparison.json'}")

Loading dataset...
Calculating stat scores...
Scoring Isolation Forest...
Scoring autoencoder...
Combining scores and selecting threshold...
{
  "random_seed": 42,
  "score_normalization": "Validation-split percentile ranks",
  "hybrid_components": [
    "z_score",
    "iqr",
    "isolation_forest",
    "autoencoder"
  ],
  "hybrid_weights": {
    "z_score": 0.25,
    "iqr": 0.25,
    "isolation_forest": 0.25,
    "autoencoder": 0.25
  },
  "validation_results": {
    "z_score": {
      "threshold": 0.5333333333333333,
      "precision": 0.45454545454545453,
      "recall": 0.20270270270270271,
      "f1": 0.2803738317757009,
      "average_precision": 0.17856191790609344,
      "false_positive_rate": 0.00042206954768213474,
      "true_negatives": 42629,
      "false_positives": 18,
      "false_negatives": 59,
      "true_positives": 15
    },
    "iqr": {
      "threshold": 0.5666666666666667,
      "precision": 0.34782608695652173,
      "recall": 0.32432432432432434,
      "f1": 0